# Professional Creators: Data Collection and Panel Construction

Collect and aggregate video data for top 30 professional creators in 2010.


In [6]:
!pip install google-api-python-client isodate -q

import pandas as pd
import isodate
import time
from googleapiclient.discovery import build

In [7]:
API_KEY = "YOUR_YOUTUBE_API_KEY"

youtube = build("youtube", "v3", developerKey=API_KEY)

START = "2009-12-01T00:00:00Z"
END = "2011-12-31T23:59:59Z"
POLICY_DATE = pd.Timestamp("2010-12-01")

1. YouTube API: Fetch seed channels

In [9]:
channels = pd.read_csv("/content/top30_2010_template.csv")

channels["channel_id"] = (
    channels["channel_id"]
    .astype(str)
    .str.strip()
)

channels = channels[
    channels["channel_id"].str.match(r"^UC[a-zA-Z0-9_-]{22}$", na=False)
].copy()

channels = channels.drop_duplicates(subset=["channel_id"]).reset_index(drop=True)

print("Seed channels:", len(channels))
channels.head()

Seed channels: 30


,input_name,channel_id
0,nigahiga,UCSAUGyc_xA8uYzaIVG6MESQ
1,RayWilliamJohnson,UCGt7X90Au6BV8rf49BiM6Dg
2,ShaneDawsonTV,UCN9wHzrHRdKVzCSeV-5RuzA
3,Fred,UC8jFPZS-p9MmVRCZi8giNcw
4,smosh,UCJ2ZDzMRgSrxmwphstrm8Ww


In [10]:
def get_upload_playlist(channel_id):
    r = youtube.channels().list(
        part="snippet,contentDetails",
        id=channel_id
    ).execute()

    if not r.get("items"):
        return None, None

    item = r["items"][0]

    channel_title = item["snippet"]["title"]
    upload_playlist = item["contentDetails"]["relatedPlaylists"]["uploads"]

    return channel_title, upload_playlist

2. YouTube API: Get video list by channel

In [11]:
def get_video_list_by_channel(upload_playlist):
    video_ids = []
    token = None

    while True:
        r = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=upload_playlist,
            maxResults=50,
            pageToken=token
        ).execute()

        for item in r.get("items", []):
            published_time = item["contentDetails"].get("videoPublishedAt")
            video_id = item["contentDetails"].get("videoId")

            if published_time and video_id:
                if START <= published_time <= END:
                    video_ids.append(video_id)

        token = r.get("nextPageToken")

        if not token:
            break

        time.sleep(0.05)

    return video_ids

3. YouTube API: Get video details

In [12]:
def get_video_details(video_ids, channel_id, channel_title):
    rows = []

    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]

        r = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(batch_ids)
        ).execute()

        for item in r.get("items", []):
            snippet = item["snippet"]
            stats = item.get("statistics", {})
            content = item["contentDetails"]

            duration_sec = isodate.parse_duration(
                content["duration"]
            ).total_seconds()

            rows.append({
                "creator_id": channel_id,
                "channel_title": channel_title,
                "video_id": item["id"],
                "timestamp": snippet["publishedAt"],
                "duration": duration_sec,
                "view_count": int(stats.get("viewCount", 0)),
                "like_count": int(stats.get("likeCount", 0)),
                "comment_count": int(stats.get("commentCount", 0))
            })

        time.sleep(0.05)

    return rows

4. Generate video-level raw dataset

In [13]:
all_rows = []

for idx, row in channels.iterrows():
    channel_id = row["channel_id"]

    try:
        channel_title, upload_playlist = get_upload_playlist(channel_id)

        if upload_playlist is None:
            print(idx, channel_id, "No upload playlist")
            continue

        video_ids = get_video_list_by_channel(upload_playlist)

        if len(video_ids) == 0:
            print(idx, channel_title, "No videos in time window")
            continue

        video_rows = get_video_details(
            video_ids=video_ids,
            channel_id=channel_id,
            channel_title=channel_title
        )

        all_rows.extend(video_rows)

        print(idx, channel_title, "videos:", len(video_rows))

        pd.DataFrame(all_rows).to_csv(
            "/content/youtube_video_raw_backup.csv",
            index=False
        )

    except Exception as e:
        if "quotaExceeded" in str(e):
            print("Quota exceeded. Stop.")
            break
        print("Error:", channel_id, e)
        time.sleep(1)

video_raw = pd.DataFrame(all_rows)

video_raw.to_csv("/content/youtube_video_raw.csv", index=False)

print("Video-level raw dataset rows:", len(video_raw))
video_raw.head()

0 nigahiga videos: 54
1 Ray William Johnson videos: 184
2 Shane Dawson TV videos: 10
3 Fred Beyer No videos in time window
4 Smosh Games No videos in time window
5 Joe Penna / MysteryGuitarMan videos: 136
6 Machinima No videos in time window
7 Shane2 No videos in time window
8 Annoying Orange videos: 124
9 sxepil No videos in time window
10 Dropout videos: 568
11 Dave Days videos: 36
12 KevJumba Archive No videos in time window
13 FAIL Blog No videos in time window
14 Universal Music Group videos: 3
15 Kassem G videos: 4
16 Michelle Phan videos: 26
17 The Key of Awesome videos: 336
18 THE STATION - Warhammer и Настольные Ролевые Игры No videos in time window
19 BuckHollywood videos: 3
20 iJustine videos: 163
21 VenetianPrincess videos: 20
22 makemebad35 videos: 40
23 Household Hacker videos: 153
24 shaycarl videos: 77
25 Kidrauhl Videos No videos in time window
26 communitychannel videos: 12
27 ExpertVillage Leaf Group No videos in time window
28 Jonas Brothers videos: 6
29 Selena Gome

,creator_id,channel_title,video_id,timestamp,duration,view_count,like_count,comment_count
0,UCSAUGyc_xA8uYzaIVG6MESQ,nigahiga,kNMjx6tnJbQ,2011-12-15T07:54:45Z,163.0,10935888,173505,25404
1,UCSAUGyc_xA8uYzaIVG6MESQ,nigahiga,cKfiurUtglA,2011-12-06T03:52:11Z,208.0,6612769,115338,20671
2,UCSAUGyc_xA8uYzaIVG6MESQ,nigahiga,amhvFQeDyGY,2011-11-24T04:55:23Z,171.0,12960581,157472,20914
3,UCSAUGyc_xA8uYzaIVG6MESQ,nigahiga,477q4Gn53tI,2011-11-19T10:17:12Z,83.0,9397298,132875,22791
4,UCSAUGyc_xA8uYzaIVG6MESQ,nigahiga,GTmhwHRQ1ow,2011-11-05T05:38:29Z,251.0,12560801,156244,18467


5. PySpark ETL logic in pandas

In [14]:
video_raw["timestamp"] = pd.to_datetime(video_raw["timestamp"])

video_raw["year_month"] = (
    video_raw["timestamp"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

video_raw["is_long_video"] = video_raw["duration"] > 900

video_raw = video_raw.drop_duplicates(subset=["video_id"])

video_raw.head()

/tmp/ipykernel_1113/1851664745.py:5: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


,creator_id,channel_title,video_id,timestamp,duration,view_count,like_count,comment_count,year_month,is_long_video
0,UCSAUGyc_xA8uYzaIVG6MESQ,nigahiga,kNMjx6tnJbQ,2011-12-15 07:54:45+00:00,163.0,10935888,173505,25404,2011-12-01,False
1,UCSAUGyc_xA8uYzaIVG6MESQ,nigahiga,cKfiurUtglA,2011-12-06 03:52:11+00:00,208.0,6612769,115338,20671,2011-12-01,False
2,UCSAUGyc_xA8uYzaIVG6MESQ,nigahiga,amhvFQeDyGY,2011-11-24 04:55:23+00:00,171.0,12960581,157472,20914,2011-11-01,False
3,UCSAUGyc_xA8uYzaIVG6MESQ,nigahiga,477q4Gn53tI,2011-11-19 10:17:12+00:00,83.0,9397298,132875,22791,2011-11-01,False
4,UCSAUGyc_xA8uYzaIVG6MESQ,nigahiga,GTmhwHRQ1ow,2011-11-05 05:38:29+00:00,251.0,12560801,156244,18467,2011-11-01,False


6. Aggregate to creator-month panel dataset

In [15]:
creator_month_panel = video_raw.groupby(
    ["creator_id", "channel_title", "year_month"],
    as_index=False
).agg(
    monthly_video_count=("video_id", "count"),
    monthly_long_video_count=("is_long_video", "sum"),
    monthly_views_sum_current=("view_count", "sum"),
    monthly_likes_sum_current=("like_count", "sum"),
    monthly_comments_sum_current=("comment_count", "sum"),
    avg_duration=("duration", "mean")
)

creator_month_panel = creator_month_panel.rename(
    columns={"year_month": "month_date"}
)

creator_month_panel.head()

,creator_id,channel_title,month_date,monthly_video_count,monthly_long_video_count,monthly_views_sum_current,monthly_likes_sum_current,monthly_comments_sum_current,avg_duration
0,UC77WzpPRrYr0W5oeFjVIqwQ,shaycarl,2009-12-01,4,0,3501386,89762,0,212.250000
1,UC77WzpPRrYr0W5oeFjVIqwQ,shaycarl,2010-02-01,7,0,18924971,182113,0,174.142857
2,UC77WzpPRrYr0W5oeFjVIqwQ,shaycarl,2010-03-01,10,0,21285765,275557,0,235.400000
3,UC77WzpPRrYr0W5oeFjVIqwQ,shaycarl,2010-04-01,11,0,9355197,149700,0,170.909091
4,UC77WzpPRrYr0W5oeFjVIqwQ,shaycarl,2010-05-01,2,0,1575379,34308,0,190.500000


7. R DID Analysis-ready variables

In [16]:
creator_month_panel["post_policy"] = (
    creator_month_panel["month_date"] >= POLICY_DATE
).astype(int)

first_long_month = video_raw[
    video_raw["is_long_video"]
].groupby("creator_id")["year_month"].min()

creator_month_panel["treated_creator"] = (
    creator_month_panel["creator_id"].isin(first_long_month.index)
).astype(int)

creator_month_panel["did"] = (
    creator_month_panel["treated_creator"] *
    creator_month_panel["post_policy"]
)

creator_month_panel.head()

,creator_id,channel_title,month_date,monthly_video_count,monthly_long_video_count,monthly_views_sum_current,monthly_likes_sum_current,monthly_comments_sum_current,avg_duration,post_policy,treated_creator,did
0,UC77WzpPRrYr0W5oeFjVIqwQ,shaycarl,2009-12-01,4,0,3501386,89762,0,212.250000,0,0,0
1,UC77WzpPRrYr0W5oeFjVIqwQ,shaycarl,2010-02-01,7,0,18924971,182113,0,174.142857,0,0,0
2,UC77WzpPRrYr0W5oeFjVIqwQ,shaycarl,2010-03-01,10,0,21285765,275557,0,235.400000,0,0,0
3,UC77WzpPRrYr0W5oeFjVIqwQ,shaycarl,2010-04-01,11,0,9355197,149700,0,170.909091,0,0,0
4,UC77WzpPRrYr0W5oeFjVIqwQ,shaycarl,2010-05-01,2,0,1575379,34308,0,190.500000,0,0,0


8. Save final datasets

In [17]:
video_raw.to_csv("/content/youtube_video_level_raw.csv", index=False)

creator_month_panel.to_csv(
    "/content/youtube_creator_month_panel.csv",
    index=False
)

print("Video-level raw rows:", len(video_raw))
print("Creator-month panel rows:", len(creator_month_panel))

Video-level raw rows: 2068
Creator-month panel rows: 351
